# Mastering GBM
Welcome to the guide on GBM.
### Kaggle Dataset:
[Titanic Dataset](https://www.kaggle.com/c/titanic)


### Why and What: Imports
**WHY:** Libraries for data manipulation.
**WHAT:** pandas, numpy, sklearn, matplotlib.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_theme(style='whitegrid')


## Part 1: Theory Recap

Gradient Boosting is the foundation upon which XGBoost and LightGBM are built.

### Real-World Analogy
Imagine playing golf. Your first shot lands far from the hole. The next shot is taken exactly from where the first landed, aiming only at the remaining distance (the residual). You repeat this until you're in the hole.

### Core Mathematics

1. **Negative Gradient (Pseudo-Residuals):**
$$ r_{im} = - \left[ \frac{\partial L(y_i, F(x_i))}{\partial F(x_i)} \right]_{F(x) = F_{m-1}(x)} $$
* $r_{im}$ : Pseudo-residual for sample $i$ at step $m$.
* $L$ : Differentiable loss function.
* $F_{m-1}(x)$ : The ensemble model's prediction up to the previous step.

2. **Model Update Rule:**
$$ F_m(x) = F_{m-1}(x) + \nu \cdot \gamma_m h_m(x) $$
* $F_m(x)$ : Updated model ensemble.
* $\nu$ : Learning rate (shrinkage factor between 0 and 1) to prevent overfitting.
* $h_m(x)$ : The new weak learner (decision tree) fitted specifically to the residuals $r_{im}$.
* $\gamma_m$ : The optimal multiplier for the new tree.


### Why and What: Loading Data
**WHY:** Real world data.
**WHAT:** Titanic.


In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
display(df.head())


### Why and What: Preprocessing
**WHY:** ML models need clean numerical data.
**WHAT:** Fillna, encode, scale.


In [ ]:
df_clean = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Fare'] = df_clean['Fare'].fillna(df_clean['Fare'].median())
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
df_clean['Sex'] = df_clean['Sex'].map({'male': 0, 'female': 1})
df_clean = pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True)
X = df_clean.drop('Survived', axis=1).values
y = df_clean['Survived'].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


## Part 2: From Scratch Implementation
### Why and What: Scratch
**WHY:** Demystify the black box.
**WHAT:** Numpy implementation.


In [ ]:
from sklearn.tree import DecisionTreeRegressor
class ScratchModel:
    def __init__(self, n=50, lr=0.1):
        self.n = n; self.lr = lr; self.trees = []
    def fit(self, X, y):
        pred = np.zeros(y.shape[0])
        for _ in range(self.n):
            p = 1 / (1 + np.exp(-pred))
            tree = DecisionTreeRegressor(max_depth=1).fit(X, y - p)
            pred += self.lr * tree.predict(X)
            self.trees.append(tree)
    def predict_proba(self, X):
        pred = np.zeros(X.shape[0])
        for t in self.trees: pred += self.lr * t.predict(X)
        return 1 / (1 + np.exp(-pred))
    def predict(self, X): return (self.predict_proba(X) >= 0.5).astype(int)


### Why and What: Evaluation
**WHY:** Verify correctness.
**WHAT:** Predict and accuracy.


In [ ]:
scratch_model = ScratchModel()
scratch_model.fit(X_train, y_train)
y_pred_custom = scratch_model.predict(X_test)
print(f"Scratch Accuracy: {accuracy_score(y_test, y_pred_custom)*100:.2f}%")


## Part 3: Library Implementation
### Why and What: Library
**WHY:** Optimized production code.
**WHAT:** Official package.


In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
model_library = GradientBoostingClassifier(n_estimators=50, random_state=42)
model_library.fit(X_train, y_train)
print('GBM Accuracy:', accuracy_score(y_test, model_library.predict(X_test)))


### Why and What: Visualizations
**WHY:** Diagnose behavior.
**WHAT:** ROC and Importances.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if hasattr(model_library, 'predict_proba'):
    fpr, tpr, _ = roc_curve(y_test, model_library.predict_proba(X_test)[:, 1])
    axes[0].plot(fpr, tpr, color='darkorange', label=f'ROC (area = {auc(fpr, tpr):.2f})')
axes[0].plot([0,1], [0,1], 'k--', label='Random Guess')
axes[0].set_title('ROC Curve')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(loc='lower right')
importances = getattr(model_library, 'feature_importances_', None)
if importances is not None:
    idx = np.argsort(importances)
    axes[1].barh(range(len(idx)), importances[idx], color='teal', label='Importance')
    axes[1].set_yticks(range(len(idx)))
    axes[1].set_yticklabels(df_clean.drop('Survived', axis=1).columns[idx])
    axes[1].legend(loc='lower right')
axes[1].set_title('Feature Importances')
axes[1].set_xlabel('Relative Importance'); axes[1].set_ylabel('Features')
plt.tight_layout()
plt.show()


## Part 4: Hyperparameter Experiments
### Why and What: Tuning
**WHY:** Optimize model.
**WHAT:** Grid search.


In [ ]:
res=[]
for lr in [0.01, 0.1]:
 for n in [10, 50, 100]:
  clf=GradientBoostingClassifier(n_estimators=n, learning_rate=lr, random_state=42)
  res.append({'n':n, 'lr':lr, 'acc':cross_val_score(clf, X_scaled, y, cv=3).mean()})
sns.lineplot(data=pd.DataFrame(res), x='n', y='acc', hue='lr')
plt.title('GBM Hyperparameters'); plt.legend(title='LR'); plt.show()


## Part 5: Interview Corner
**Q: Why residuals?**
**A:** It represents the negative gradient of the loss, allowing functional gradient descent.


## Key Takeaways
- Additive.
- Residual fitting.
- Shrinkage.
- Prone to overfitting without bounds.
- Versatile.
